## Import libraries

In [1]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [19]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

## Step 1a - Indexing (Document Ingestion)

In [13]:
from youtube_transcript_api import YouTubeTranscriptApi

# YouTube video ID (not full URL)
video_id = "T-D1OfcDW1M"

# Create API object (new versions require an instance)
api = YouTubeTranscriptApi()

try:
    # 1️⃣ Fetch list of all available transcripts (languages, auto/manual)
    transcripts = api.list(video_id)

    # 2️⃣ Select English transcript (manual if available)
    transcript = transcripts.find_transcript(["en"])

    # 3️⃣ Download transcript data
    # Returns a list of FetchedTranscriptSnippet objects
    data = transcript.fetch()

    # 4️⃣ Extract only text from each snippet and join into one string
    # Use attribute access (.text), not dictionary indexing
    text = " ".join(chunk.text for chunk in data)

    # 5️⃣ Print final transcript
    print(text)

except Exception as e:
    # Runs if transcript is disabled, unavailable, or language not found
    print("Transcript not available:", e)


Large language models. They are everywhere. They get some things amazingly right and other things very interestingly wrong. My name is Marina Danilevsky. I am a Senior Research Scientist here at IBM Research. And I want to tell you about a framework to help large language models be more accurate and more up to date: Retrieval-Augmented Generation, or RAG. Let's just talk about the "Generation" part for a minute. So forget the "Retrieval-Augmented". So the generation, this refers to large language models, or LLMs, that generate text in response to a user query, referred to as a prompt. These models can have some undesirable behavior. I want to tell you an anecdote to illustrate this. So my kids, they recently asked me this question: "In our solar system, what planet has the most moons?" And my response was, “Oh, that's really great that you're asking this question. I loved space when I was your age.” Of course, that was like 30 years ago. But I know this! I read an article and the artic

In [14]:
for chunk in data:
    print(chunk.start, chunk.text)


0.06 Large language models. They are everywhere.
2.632 They get some things amazingly right
5.267 and other things very interestingly wrong.
7.819 My name is Marina Danilevsky.
9.578 I am a Senior Research Scientist here at IBM Research.
12.314 And I want to tell you about a framework to help large language models
16.549 be more accurate and more up to date:
18.648 Retrieval-Augmented Generation, or RAG.
22.68 Let's just talk about the "Generation" part for a minute.
24.784 So forget the "Retrieval-Augmented".
26.8 So the generation, this refers to large language models, or LLMs,
31.077 that generate text in response to a user query, referred to as a prompt.
36.0 These models can have some undesirable behavior.
38.269 I want to tell you an anecdote to illustrate this.
41.284 So my kids, they recently asked me this question:
44.44 "In our solar system, what planet has the most moons?"
48.713 And my response was, “Oh, that's really great that you're asking this question. I loved space wh

## Step 1b - Indexing (Text Splitting)

In [15]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# transcript/text should be a STRING
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.create_documents([text])  # pass list of strings


In [17]:
len(chunks)

8

In [18]:
chunks[5]

Document(metadata={}, page_content='"No, no, no." "First, go and retrieve\xa0relevant content." "Combine that with the user\'s question and only then generate the\xa0answer." So the prompt now has three parts: the instruction to pay attention to, the retrieved\xa0content, together with the user\'s question. Now give a response. And in fact, now you can give\xa0evidence for why your response was what it was.\xa0\xa0 So now hopefully you can see, how does RAG help the two LLM challenges that I had mentioned before?\xa0\xa0 So first of all, I\'ll start with the out of\xa0date part. Now, instead of having to retrain your model, if new information comes up, like, hey,\xa0we found some more moons-- now to Jupiter again, maybe it\'ll be Saturn again in the future. All\xa0you have to do is you augment your data store with new information, update information. So now the next time that a user comes and asks the question, we\'re ready. We just go ahead and retrieve the most up to date information

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [20]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small",api_key=api_key)
vector_store = FAISS.from_documents(chunks, embeddings)

In [21]:
vector_store.index_to_docstore_id

{0: 'dcd3d74a-9f43-4cde-83d1-6218c8c6cbb4',
 1: '3b6f0093-dbd0-407e-9870-48897020edd2',
 2: 'e8ee28d8-9d32-4369-87d0-4cedf6eb49ea',
 3: '86517fc3-b44b-4a3a-be2a-5c21dfa01fa7',
 4: 'a27cde5d-aaf2-4975-8ef5-82847b490cf8',
 5: '5e55211c-77fb-40a9-adf9-9a1ecd3c3b96',
 6: 'dd41eb77-064e-4204-8157-0e25dd80c18d',
 7: '710c7e90-01a4-4439-ae07-291cd7e4d422'}

In [22]:
vector_store.get_by_ids(['a27cde5d-aaf2-4975-8ef5-82847b490cf8'])

[Document(id='a27cde5d-aaf2-4975-8ef5-82847b490cf8', metadata={}, page_content='are adding a content store. This could be open like the internet. This\xa0can be closed like some collection of documents, collection of policies, whatever. The point,\xa0though, now is that the LLM first goes and talks to the content store and says,\xa0“Hey, can you retrieve for me information that is relevant to what the user\'s\xa0query was?” And now, with this retrieval-augmented answer, it\'s not Jupiter anymore. We know that\xa0it is Saturn. What does this look like? Well, first user prompts the LLM\xa0with their question. They say, this is what my question was. And originally,\xa0if we\'re just talking to a generative model, the generative model says, “Oh, okay, I know\xa0the response. Here it is. Here\'s my response.”\xa0\xa0 But now in the RAG framework, the generative\xa0model actually has an instruction that says, "No, no, no." "First, go and retrieve\xa0relevant content." "Combine that with the 

## Step 2 - Retrieval

In [25]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [26]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002642F428830>, search_kwargs={'k': 3})

In [27]:
retriever.invoke('What is rag')

[Document(id='a27cde5d-aaf2-4975-8ef5-82847b490cf8', metadata={}, page_content='are adding a content store. This could be open like the internet. This\xa0can be closed like some collection of documents, collection of policies, whatever. The point,\xa0though, now is that the LLM first goes and talks to the content store and says,\xa0“Hey, can you retrieve for me information that is relevant to what the user\'s\xa0query was?” And now, with this retrieval-augmented answer, it\'s not Jupiter anymore. We know that\xa0it is Saturn. What does this look like? Well, first user prompts the LLM\xa0with their question. They say, this is what my question was. And originally,\xa0if we\'re just talking to a generative model, the generative model says, “Oh, okay, I know\xa0the response. Here it is. Here\'s my response.”\xa0\xa0 But now in the RAG framework, the generative\xa0model actually has an instruction that says, "No, no, no." "First, go and retrieve\xa0relevant content." "Combine that with the 

## Step 3 - Augmentation

In [29]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2,api_key=api_key)

In [30]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [31]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [32]:
retrieved_docs

[Document(id='e8ee28d8-9d32-4369-87d0-4cedf6eb49ea', metadata={}, page_content="interacting with large language models. They’re LLM\xa0challenges. Now, what would have happened if I'd taken a beat and first gone and looked\xa0up the answer on a reputable source like NASA? Well, then I would have been able to say, “Ah,\xa0okay! So the answer is Saturn with 146 moons.” And in fact, this keeps changing because scientists\xa0keep on discovering more and more moons. So I have now grounded my answer in something more"),
 Document(id='3b6f0093-dbd0-407e-9870-48897020edd2', metadata={}, page_content="And my response was, “Oh, that's really great that you're asking this question. I loved\xa0space when I was your age.” Of course, that was like 30 years ago. But I know this! I read an\xa0article and the article said that it was Jupiter and 88 moons. So that's the answer. Now, actually,\xa0there's a couple of things wrong with my answer. First of all, I have no source to support what\xa0I'm saying

In [33]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"interacting with large language models. They’re LLM\xa0challenges. Now, what would have happened if I'd taken a beat and first gone and looked\xa0up the answer on a reputable source like NASA? Well, then I would have been able to say, “Ah,\xa0okay! So the answer is Saturn with 146 moons.” And in fact, this keeps changing because scientists\xa0keep on discovering more and more moons. So I have now grounded my answer in something more\n\nAnd my response was, “Oh, that's really great that you're asking this question. I loved\xa0space when I was your age.” Of course, that was like 30 years ago. But I know this! I read an\xa0article and the article said that it was Jupiter and 88 moons. So that's the answer. Now, actually,\xa0there's a couple of things wrong with my answer. First of all, I have no source to support what\xa0I'm saying. So even though I confidently said “I read an article, I know the answer!”, I'm not\xa0sourcing it. I'm giving the answer off the top of my head. And also, I 

In [34]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [35]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      interacting with large language models. They’re LLM\xa0challenges. Now, what would have happened if I'd taken a beat and first gone and looked\xa0up the answer on a reputable source like NASA? Well, then I would have been able to say, “Ah,\xa0okay! So the answer is Saturn with 146 moons.” And in fact, this keeps changing because scientists\xa0keep on discovering more and more moons. So I have now grounded my answer in something more\n\nAnd my response was, “Oh, that's really great that you're asking this question. I loved\xa0space when I was your age.” Of course, that was like 30 years ago. But I know this! I read an\xa0article and the article said that it was Jupiter and 88 moons. So that's the answer. Now, actually,\xa0there's a couple of things wrong with my answer. First of all, I have no sourc

## Step 4 - Generation

In [36]:
answer = llm.invoke(final_prompt)
print(answer.content)

I don't know.


### Next question

In [37]:
question          = "what is Rag?"
retrieved_docs    = retriever.invoke(question)
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
final_prompt = prompt.invoke({"context": context_text, "question": question})
answer = llm.invoke(final_prompt)
print(answer.content)

Retrieval-Augmented Generation (RAG) is a framework designed to help large language models (LLMs) be more accurate and up to date. In this framework, when a user prompts the LLM with a question, the model first retrieves relevant content from a content store before generating a response. This process involves combining the retrieved information with the user's query to produce a more informed and accurate answer. RAG addresses the potential issue of the LLM generating incorrect responses by ensuring it has access to high-quality grounding information through the retrieval process.


## Building a Chain

In [38]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [39]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [40]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [41]:
parallel_chain.invoke('who is Marina Danilevsky?')

{'context': 'Large language models. They are everywhere. They get some things amazingly right and other things very interestingly wrong. My name\xa0is Marina Danilevsky. I am a Senior Research Scientist here at IBM Research. And I want\xa0to tell you about a framework to help large language models be more accurate and more up to\xa0date: Retrieval-Augmented Generation, or RAG. Let\'s just talk about the "Generation" part for a\xa0minute. So forget the "Retrieval-Augmented". So the\xa0generation, this refers to large language models,\xa0or LLMs, that generate text in response to a user query, referred to as a prompt. These\xa0models can have some undesirable behavior. I want to tell you an anecdote to illustrate this. So my kids, they recently asked me this question: "In our solar system, what planet has the most\xa0moons?" And my response was, “Oh, that\'s really great that you\'re asking this question. I loved\xa0space when I was your age.” Of course, that was like 30 years ago. But I

In [42]:
parser = StrOutputParser()

In [43]:
main_chain = parallel_chain | prompt | llm | parser

In [44]:
main_chain.invoke('Can you summarize the video')

"I don't know."